## The Union–Find Data Structure
The **union–find data structure** maintains a partition of the set $1..n$ and supports two operations: 
1) The function `find(i)` returns the representative of the block containing `i`.Thus, testing whether two elements are in the same block amounts to comparing
their respective representatives. 
2) An operation `link(i, j)` applied to representatives of different blocks joins the blocks.

Initially, each element is a block on its own. Each block chooses one of its elements as its representative; the choice is made by the data structure and not by the user. 

```mermaid
classDiagram
class UnionFind {
    +link(node1: Node, node2: Node)
    +find(node: Node) Node
    +union(node1: Node, node2: Node)
}
```

## Optimization used
### Union by rank
Every representative stores a nonnegative integer, which we call its **rank**. 

Initially, every element is a representative and has rank zero. When we link two representatives and their ranks are different, we make the representative of smaller rank a child of the representative of larger rank.

When their ranks are the same, the choice of the parent is arbitrary; however, we
increase the rank of the new root.

### Path compression
This ensures that a chain of parent references is never traversed twice. Rather, all nodes visited during an operation `find(i)` redirect their parent pointers directly to the representative of `i`.

In [ ]:
# Union Find DS implementation
from typing import Iterable, Union
from treelib import Tree

Node = Union[str, int]


class UnionFind:
    def __init__(self, elements: Iterable[Node]) -> None:
        self.parents = {e: e for e in elements}     # represantatives
        self.ranks = {e: 0 for e in elements}       # rank of representatives ONLY!

    def find(self, elem: Node) -> Node:
        parent = self.parents[elem]
        if parent == elem:
            return elem
        
        else:
            grandpa = self.find(parent)             # Path compression
            self.parents[elem] = grandpa
            return grandpa
        
    def link(self, e1: Node, e2: Node) -> None:
        if self.find(e1) == self.find(e2):          # same block
            return None
        
        if self.ranks[e1] < self.ranks[e2]:         # Union by rank
            self.parents[e1] = e2

        else:
            self.parents[e2] = e1

            if self.ranks[e1] == self.ranks[e2]:    # increment the rank
                self.ranks[e2] += 1

    def union(self, e1: Node, e2: Node) -> None:
        if self.find(e1) != self.find(e2):          # same block
            self.link(self.find(e1), self.find(e2))

    def to_console(self) -> None:
        forest = {}
        def add_to_tree(v, p):
            if v == p:
                if v not in forest:
                    tree = Tree()
                    tree.create_node(v, v)
                    forest[v] = tree
                return forest[v]

            else:
                tree = add_to_tree(p, self.parents[p])
                if v not in tree:
                    tree.create_node(v, v, parent=p)
                return tree

        for v, p in self.parents.items():
            add_to_tree(v, p)

        for t in forest.values():
            print(t)

In [ ]:
# Testing 

nodes = ["A", "B", "C", "D", "E"]
uf = UnionFind(nodes)
uf.union("A", "C")
uf.union("E", "C")
uf.union("B", "D")
uf.union("D", "C")

print(f"{uf.parents=}, {uf.ranks=}")
uf.to_console()
uf.find("C")
uf.to_console()


# Kruskal’s Algorithm

This algorithm does not consider the vertices directly at all, but builds a minimal spanning tree by considering and adding
edges as follows: Assume that we already have a collection of edges `T`. 

Then, from all the edges not yet in `T`, choose one with minimal weight such that its addition to `T` does not produce a circle, and add that to `T`. 
If we start with `T` being the empty set, and continue until no more edges can be added, a minimal spanning tree will be produced. 

This approach is known as Kruskal's algorithm.


In [ ]:
# Input: A directed graph with weight matrix `weight'.
# Output: Spanning Tree.

graph = {
    'A': {'B': 4, 'C': 4},
    'B': {'A': 4, 'C': 2},
    'C': {'A': 4, 'B': 2, 'D': 3, 'E': 1, 'F': 6},
    'D': {'C': 3, 'F': 2},
    'E': {'C': 1, 'F': 3},
    'F': {'C': 6, 'D': 2, 'E': 3},
}

```mermaid
flowchart LR
    A((A))
    A-- 4 -->B
    A-- 4 -->C

    B((B))
    B-- 4 -->A
    B-- 2 -->C

    C((C))
    C-- 4 -->A
    C-- 2 -->B
    C-- 3 -->D
    C-- 1 -->E
    C-- 6 -->F

    D((D))
    D-- 3 -->C
    D-- 2 -->F

    E((E))
    E-- 1 -->C
    E-- 3 -->F

    F((F))
    F-- 6 -->C
    F-- 2 -->D
    F-- 3 -->E
```

In [ ]:
# Min-Priority Queue

class MinHeap:
    def __init__(self):
        self.heap = []

    # return the number of nodes in a heap
    def size(self):
            return len(self.heap)

    # check if the heap is empty
    def is_empty(self):
        return len(self.heap) == 0

    # string representation of a heap
    def __str__(self):
        return str(self.heap)

    # swap the heap values
    def swap(self, i, j):
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    # calculate the index of a node's parent
    def parent(self, index):
        return (index - 1) // 2

    # calculate the index of a node's left child
    def left_child(self, index):
        return 2 * index + 1

    # calculate the index of a node's right_child
    def right_child(self, index):
        return 2 * index + 2

    # check if a node at a given index has a parent
    def has_parent(self, index):
        return self.parent(index) >= 0

    # check if a node at a given index has a left child
    def has_left_child(self, index):
        return self.left_child(index) < len(self.heap)

    # check if a node at a given index has a right child
    def has_right_child(self, index):
        return self.right_child(index) < len(self.heap)
       
    # heapify 
    def heapify(self, array):
        self.heap = array
        for i in range(len(self.heap) - 1, -1, -1):
            if  self.has_left_child(i):
                self.heapify_down(i)
        
    # heapify down
    def heapify_down(self, index):
        if not self.has_left_child(index):
            return
        smaller_child_index = self.left_child(index)
        if self.has_right_child(index): 
            if self.heap[self.right_child(index)] < self.heap[smaller_child_index]:
                smaller_child_index = self.right_child(index)
        if self.heap[index] > self.heap[smaller_child_index]:
            self.swap(index, smaller_child_index)
        if self.has_left_child(smaller_child_index):
            self.heapify_down(smaller_child_index)
    
    # heapify up
    def heapify_up(self, index):
        if self.has_parent(index): 
            parent_index = self.parent(index)
            # compare the value to its parent and swap if necessary
            if self.heap[index] < self.heap[self.parent(index)]:
                parent_index = self.parent(index)
                self.swap(index, parent_index)
                # run the code for parent                 
                if self.has_parent(parent_index):
                    self.heapify_up(parent_index)

    # insert into heap
    def insert(self, value):
        self.heap.append(value)
        index = len(self.heap) - 1
        while self.has_parent(index) and self.heap[index] < self.heap[self.parent(index)]:
            parent_index = self.parent(index)
            self.swap(index, parent_index)
            index = parent_index
    
    # extract min from heap
    def extract_min(self):
        min_value = self.heap[0]
        if self.size() <= 1:
            self.heap = []
            return min_value
        self.heap = [self.heap[-1]] + self.heap[1:-1]
        index = 0
        while True:
            if not self.has_left_child(index):
                break
            smaller_child_index = self.left_child(index)
            if self.has_right_child(index): 
                if self.heap[self.right_child(index)] < self.heap[self.left_child(index)]:
                    smaller_child_index = self.right_child(index)
            if self.heap[index] > self.heap[smaller_child_index]:
                self.swap(index, smaller_child_index)
            else:
                break 
            index = smaller_child_index
        return min_value
    
    # peek
    def peek(self):
        return self.heap[0]
    

class PriorityQueue(MinHeap):
    def __init__(self):
        super().__init__()

    def insert(self, priority, value):
        return super().insert((priority, value))
    
    # change the priority (key) of an element already in the priority queue
    def change_key(self, value, new_priority):
        # search for value
        for i in range(self.size()):
            # if value is found
            if self.heap[i][1] == value:
                # change priority
                old_priority = self.heap[i][0]
                self.heap[i] = (new_priority, value)
                # heapify after changing priority to maintain heap property.
                # we can simply call heapify but heapify_up/down is more efficient
                if new_priority < old_priority:
                    self.heapify_up(i)
                else:
                    self.heapify_down(i)
                return


In [ ]:
# Kruskal's Algorithm with UnionFind
from pprint import pprint
from typing import Dict


def kruskal(graph: Dict[str, Dict[str, int]]):
    priority_queue = PriorityQueue()
    nodes = []
    for v, adj_list in graph.items():                   # Inserting edges to Priority Queue
        nodes.append(v)
        for u, w in adj_list.items():
            priority_queue.insert(w, (v, u))

    union_find = UnionFind(nodes)                       # Adding all nodes to UnionFind
    total = 0
    span_tree = {}
    while not priority_queue.is_empty():
        w, edge = priority_queue.extract_min()          # Getting edge
        v, u = edge

        if union_find.find(v) != union_find.find(u):    # Checking for loops
            if v not in span_tree:
                span_tree[v] = {}
            span_tree[v][u] = w
            total += w

            union_find.union(v, u)
            print(f"Edge {v}-({w})->{u} is added to min spanning tree.")
        else:
            print(f"Edge {v}-({w})->{u}'s forming a loop and is not added to min spanning tree.")

    print(f"Total spanning tree weight is {total}")
    return span_tree

spanning_tree = kruskal(graph)


In [ ]:
# Graph to mermaid

mermaid = "flowchart LR\n"

for v, adj_list in spanning_tree.items():
    # Stating Node
    mermaid += " " * 4
    mermaid += f"{v}(({v}))\n"

    # adding Edges
    for u, w in adj_list.items():
        mermaid += " " * 4
        mermaid += f"{v}-- {w} -->{u}\n"

    mermaid += "\n"

print(mermaid)

```mermaid
flowchart LR
    C((C))
    C-- 1 -->E
    C-- 3 -->D

    B((B))
    B-- 2 -->C

    D((D))
    D-- 2 -->F

    A((A))
    A-- 4 -->B
```